# Data Download: Replicating Gu, Kelly & Xiu (2020)
## "Empirical Asset Pricing via Machine Learning" — Review of Financial Studies

This notebook downloads and prepares the data needed for the cross-sectional replication.

**Data components:**
1. **94 firm-level characteristics** (from Dacheng Xiu's website or JKP/WRDS)
2. **Monthly stock returns** (from CRSP via WRDS)
3. **8 macroeconomic predictors** (Welch & Goyal 2008)
4. **74 SIC-2 industry dummies** (constructed from CRSP SIC codes)

**Total covariates:** 94 × (8 + 1) + 74 = **920** baseline signals

---

## 0. Install Dependencies

In [2]:
# Run this cell once to install required packages
!pip install requests tqdm

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
Using cached idna-3.11-py3-none-any.whl (71 kB)
Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)
Using cached certifi-2026.2.25-py3-none-any.whl (153 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [requests]


In [3]:
import pandas as pd
import numpy as np
import os
import io
import zipfile
import requests
import warnings
warnings.filterwarnings('ignore')

# Create a data directory
DATA_DIR = './data'
os.makedirs(DATA_DIR, exist_ok=True)

print('Setup complete.')

Setup complete.


---
## 1. Firm-Level Characteristics (94 predictors)

### Direct download from Dacheng Xiu's homepage 
The authors provide the pre-computed dataset of 94 lagged firm characteristics at:

**`https://dachxiu.chicagobooth.edu/download/datashare.zip`**

This file is ~3-4 GB and contains `datashare.csv` with:
- `DATE`: end of month (YYYYMMDD), already lag-adjusted
- `permno`: CRSP permanent company number
- 94 columns of firm characteristics

> **Note from the readme:** *"In this dataset, we've already adjusted the lags. (e.g. When DATE=19570329, you can use the monthly RET at 195703 as the response variable.)"*

In [5]:
# ====================================================
# OPTION A: Download characteristics from Xiu's website
# ====================================================
# WARNING: This file is ~3-4 GB. Make sure you have enough disk space and a stable connection.

url_chars = 'https://dachxiu.chicagobooth.edu/download/datashare.zip'
zip_path = os.path.join(DATA_DIR, 'datashare.zip')
csv_path = os.path.join(DATA_DIR, 'datashare.csv')

if not os.path.exists(csv_path):
    print('Downloading firm characteristics from Dacheng Xiu homepage...')
    print('This may take 10-20 minutes depending on your connection.')
    
    # Stream download to handle large file
    with requests.get(url_chars, stream=True, timeout=1200) as r:
        r.raise_for_status()
        total_size = int(r.headers.get('content-length', 0))
        downloaded = 0
        with open(zip_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192*100):
                f.write(chunk)
                downloaded += len(chunk)
                if total_size > 0:
                    pct = downloaded / total_size * 100
                    print(f'\rProgress: {pct:.1f}% ({downloaded/1e9:.2f} GB)', end='')
    print('\nDownload complete. Extracting...')
    
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATA_DIR)
    print('Extraction complete.')
    
    # Clean up zip
    os.remove(zip_path)
else:
    print(f'File already exists: {csv_path}')

This may take 10-20 minutes depending on your connection.
Progress: 100.0% (1.56 GB)
Download complete. Extracting...
Extraction complete.


In [6]:
# Load the characteristics data
print('Loading characteristics data (this may take a minute)...')
chars = pd.read_csv(csv_path, low_memory=False)

# Basic formatting
chars.rename(columns={'DATE': 'date', 'permno': 'permno'}, inplace=True)
chars['date'] = pd.to_datetime(chars['date'], format='%Y%m%d')
chars['date'] = chars['date'].dt.to_period('M')  # Convert to monthly period

print(f'Shape: {chars.shape}')
print(f'Date range: {chars["date"].min()} to {chars["date"].max()}')
print(f'Unique stocks (permno): {chars["permno"].nunique()}')
print(f'\nCharacteristics ({chars.shape[1] - 2} features):')
print([c for c in chars.columns if c not in ['date', 'permno']])

Loading characteristics data (this may take a minute)...
Shape: (4117300, 97)
Date range: 1957-01 to 2021-12
Unique stocks (permno): 32793

Characteristics (95 features):
['mvel1', 'beta', 'betasq', 'chmom', 'dolvol', 'idiovol', 'indmom', 'mom1m', 'mom6m', 'mom12m', 'mom36m', 'pricedelay', 'turn', 'absacc', 'acc', 'age', 'agr', 'bm', 'bm_ia', 'cashdebt', 'cashpr', 'cfp', 'cfp_ia', 'chatoia', 'chcsho', 'chempia', 'chinv', 'chpmia', 'convind', 'currat', 'depr', 'divi', 'divo', 'dy', 'egr', 'ep', 'gma', 'grcapx', 'grltnoa', 'herf', 'hire', 'invest', 'lev', 'lgr', 'mve_ia', 'operprof', 'orgcap', 'pchcapx_ia', 'pchcurrat', 'pchdepr', 'pchgm_pchsale', 'pchquick', 'pchsale_pchinvt', 'pchsale_pchrect', 'pchsale_pchxsga', 'pchsaleinv', 'pctacc', 'ps', 'quick', 'rd', 'rd_mve', 'rd_sale', 'realestate', 'roic', 'salecash', 'saleinv', 'salerec', 'secured', 'securedind', 'sgr', 'sin', 'sp', 'tang', 'tb', 'aeavol', 'cash', 'chtx', 'cinvest', 'ear', 'nincr', 'roaq', 'roavol', 'roeq', 'rsup', 'stdacc',

In [7]:
# Quick summary stats
chars.describe().T.head(20)

,count,mean,std,min,25%,50%,75%,max
permno,4117300.0,5.580153e+04,2.803699e+04,10000.000000,27748.000000,61321.000000,81002.000000,9.343600e+04
mvel1,4114230.0,1.794745e+06,1.425696e+07,0.000000,22628.333804,99787.901182,500489.379515,2.711977e+09
beta,3716736.0,1.017846e+00,6.469574e-01,-0.799580,0.551429,0.948765,1.393751,3.887420e+00
betasq,3716736.0,1.459436e+00,1.727259e+00,0.000000,0.309960,0.903993,1.946645,1.511203e+01
chmom,3778195.0,1.419177e-03,5.388882e-01,-8.461765,-0.237329,-0.005600,0.231787,7.783077e+00
dolvol,3765378.0,1.103710e+01,2.997975e+00,-3.060271,8.932708,10.910054,13.119557,1.945995e+01
idiovol,3716736.0,6.000247e-02,3.719688e-02,0.000000,0.033552,0.050516,0.076566,2.835521e-01
indmom,4117276.0,1.290215e-01,2.796336e-01,-0.759503,-0.039005,0.105214,0.255788,2.748655e+00
mom1m,4085562.0,9.047752e-03,1.470126e-01,-0.736589,-0.060606,0.000000,0.065848,2.000000e+00
mom6m,3960127.0,4.944756e-02,3.507820e-01,-1.000000,-0.133818,0.022727,0.180602,7.533333e+00


---
## 2. Monthly Stock Returns from CRSP

We need CRSP monthly returns to compute excess returns (the target variable). 
This requires WRDS access.

**From the paper (p. 2248):**
> We obtain monthly total individual equity returns from CRSP for all firms listed in the NYSE, AMEX, and NASDAQ. Our sample begins in March 1957.

In [8]:
import wrds

# Connect to WRDS
wrds_db = wrds.Connection()
print('Connected to WRDS.')

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Connected to WRDS.


In [9]:
# ====================================================
# Download CRSP Monthly Stock File
# ====================================================
# We get: permno, date, ret (total return), shrout, prc (price), 
# exchcd (exchange code), siccd (SIC code), shrcd (share code)

crsp_query = """
    SELECT a.permno, a.date, a.ret, a.retx, a.prc, a.shrout, a.vol,
           b.exchcd, b.siccd, b.shrcd
    FROM crsp.msf AS a
    LEFT JOIN crsp.msenames AS b
        ON a.permno = b.permno
        AND a.date >= b.namedt
        AND a.date <= b.nameendt
    WHERE a.date >= '1957-03-01'
      AND a.date <= '2021-12-31'
      AND b.exchcd IN (1, 2, 3)       -- NYSE, AMEX, NASDAQ
      AND b.shrcd IN (10, 11)          -- Common shares only
"""

print('Downloading CRSP monthly stock data...')
crsp_monthly = wrds_db.raw_sql(crsp_query, date_cols=['date'])
print(f'CRSP data shape: {crsp_monthly.shape}')
print(f'Date range: {crsp_monthly["date"].min()} to {crsp_monthly["date"].max()}')
print(f'Unique permnos: {crsp_monthly["permno"].nunique()}')

CRSP data shape: (3352317, 10)
Date range: 1957-03-29 00:00:00 to 2021-12-31 00:00:00
Unique permnos: 25845


In [10]:
# ====================================================
# Download CRSP delisting returns
# ====================================================
# Important: incorporate delisting returns to avoid survivorship bias

delist_query = """
    SELECT permno, dlstdt AS date, dlret, dlstcd
    FROM crsp.msedelist
    WHERE dlstdt >= '1957-03-01'
      AND dlstdt <= '2021-12-31'
"""

print('Downloading delisting returns...')
crsp_delist = wrds_db.raw_sql(delist_query, date_cols=['date'])
print(f'Delisting data shape: {crsp_delist.shape}')

Delisting data shape: (26414, 4)


In [11]:
# ====================================================
# Merge delisting returns with monthly returns
# ====================================================

# Align delisting date to end of month
crsp_delist['date'] = crsp_delist['date'] + pd.offsets.MonthEnd(0)
crsp_monthly['date'] = crsp_monthly['date'] + pd.offsets.MonthEnd(0)

# Merge
crsp = crsp_monthly.merge(crsp_delist[['permno', 'date', 'dlret', 'dlstcd']], 
                           on=['permno', 'date'], how='left')

# Adjust returns for delisting
# If dlret is missing and dlstcd indicates performance-related delisting, use -0.30
crsp['dlret'] = crsp['dlret'].fillna(0)
crsp.loc[
    (crsp['dlret'].isna()) & (crsp['dlstcd'].isin([500, 520, 551, 552, 560, 574])),
    'dlret'
] = -0.30

# Compute adjusted return: (1 + ret) * (1 + dlret) - 1
crsp['ret_adj'] = (1 + crsp['ret'].fillna(0)) * (1 + crsp['dlret'].fillna(0)) - 1

# Market equity (in millions)
crsp['me'] = np.abs(crsp['prc']) * crsp['shrout'] / 1000

print(f'Merged CRSP shape: {crsp.shape}')
crsp.head()

Merged CRSP shape: (3352317, 14)


,permno,date,ret,retx,prc,shrout,vol,exchcd,siccd,shrcd,dlret,dlstcd,ret_adj,me
0,10000,1986-01-31,<NA>,<NA>,-4.375,3680.0,1771.0,3,3990,10,0.0,<NA>,0.0,16.1
1,10000,1986-02-28,-0.257143,-0.257143,-3.25,3680.0,828.0,3,3990,10,0.0,<NA>,-0.257143,11.96
2,10000,1986-03-31,0.365385,0.365385,-4.4375,3680.0,1078.0,3,3990,10,0.0,<NA>,0.365385,16.33
3,10000,1986-04-30,-0.098592,-0.098592,-4.0,3793.0,957.0,3,3990,10,0.0,<NA>,-0.098592,15.172
4,10000,1986-05-31,-0.222656,-0.222656,-3.10938,3793.0,1074.0,3,3990,10,0.0,<NA>,-0.222656,11.793878


In [ ]:
# ====================================================
# Download risk-free rate from CRSP
# ====================================================

rf_query = """
    SELECT caldt AS date, t30ret AS rf
    FROM crsp.mcti
    WHERE caldt >= '1957-03-01'
      AND caldt <= '2021-12-31'
"""

print('Downloading risk-free rate...')
rf = wrds_db.raw_sql(rf_query, date_cols=['date'])
rf['date'] = rf['date'] + pd.offsets.MonthEnd(0)
print(f'Risk-free rate data shape: {rf.shape}')

Risk-free rate data shape: (778, 2)


In [13]:
# ====================================================
# Compute excess returns
# ====================================================

crsp = crsp.merge(rf, on='date', how='left')
crsp['ret_excess'] = crsp['ret_adj'] - crsp['rf']

print(f'Excess return stats:')
print(crsp['ret_excess'].describe())

Excess return stats:
count    3352317.0
mean      0.007915
std       0.180717
min      -1.014149
25%      -0.068691
50%      -0.004029
75%       0.066491
max      23.996941
Name: ret_excess, dtype: Float64


In [ ]:
# Convert date to monthly period for merging with characteristics
crsp['date'] = crsp['date'].dt.to_period('M')

# Keep relevant columns
crsp = crsp.dropna(subset=['ret_excess'])

print(f'Clean CRSP shape: {crsp.shape}')
crsp.head()

Clean CRSP shape: (3352317, 6)


,permno,date,ret_excess,me,siccd,exchcd
0,10000,1986-01,-0.004865,16.1,3990,3
1,10000,1986-02,-0.262439,11.96,3990,3
2,10000,1986-03,0.359422,16.33,3990,3
3,10000,1986-04,-0.103933,15.172,3990,3
4,10000,1986-05,-0.22759,11.793878,3990,3


In [21]:
# Data to CSV

crsp.to_csv("data/Monthly_returns_crps.csv", index=False)
rf.to_csv("data/risk_free_rate.csv", index=False)


## Variable definitions in Monthly_returns_crsp

- **permno**: Permanent CRSP identifier for each stock.
- **date**: Monthly observation date used for merging across datasets.
- **ret**: Monthly stock return including distributions such as dividends.
- **retx**: Monthly stock return excluding distributions.
- **prc**: Stock price at the end of the month.
- **shrout**: Number of shares outstanding.
- **vol**: Trading volume during the month.
- **exchcd**: Exchange code indicating where the stock is listed (e.g. NYSE, AMEX, NASDAQ).
- **siccd**: Standard Industrial Classification (SIC) code identifying the firm’s industry.
- **shrcd**: Share code indicating the type of security.
- **dlret**: Delisting return, capturing the return associated with a stock’s removal from the exchange.
- **dlstcd**: Delisting code indicating the reason for delisting.
- **ret_adj**: Delisting-adjusted return, computed as \((1+\text{ret})(1+\text{dlret})-1\).
- **me**: Market equity (market capitalization), computed as \(|prc| \times shrout / 1000\).
- **rf**: Monthly risk-free rate from CRSP.
- **ret_excess**: Excess stock return over the risk-free rate, computed as \(\text{ret\_adj} - \text{rf}\).

---
## 3. Macroeconomic Predictors (Welch & Goyal 2008)

The paper uses 8 aggregate time-series variables:
1. **dp** — Dividend-price ratio (log)
2. **ep** — Earnings-price ratio (log)
3. **bm** — Book-to-market ratio
4. **ntis** — Net equity expansion
5. **tbl** — Treasury bill rate
6. **tms** — Term spread
7. **dfy** — Default spread
8. **svar** — Stock variance

These are available from Amit Goyal's website.

In [ ]:
# ====================================================
# Download Welch & Goyal (2008) macro predictors
# ====================================================

# Download data in: url_macro = https://sites.google.com/view/agoyal145/home'


macro_raw = pd.read_excel(
        'data/PredictorData.xlsx',
        sheet_name='Monthly'
    )
print(macro_raw.head())

Attempting to download macro predictors from Goyal website...
Auto-download failed: [Errno 2] No such file or directory: 'PredictorData.xlsx'

>>> MANUAL STEP REQUIRED: <<<
1. Go to: https://sites.google.com/view/agoyal145/home
2. Download the monthly data Excel file
3. Save it as: ./data/PredictorData.xlsx
4. Re-run this cell.


In [32]:
macro_raw = pd.read_excel(
        'data/PredictorData.xlsx',
        sheet_name='Monthly'
    )
print(macro_raw.head())

   yyyymm  Index   D12  E12  b/m  tbl  AAA  BAA  lty  ntis     Rfree  infl  \
0  187101   4.44  0.26  0.4  NaN  NaN  NaN  NaN  NaN   NaN       NaN   NaN   
1  187102   4.50  0.26  0.4  NaN  NaN  NaN  NaN  NaN   NaN  0.004967   NaN   
2  187103   4.61  0.26  0.4  NaN  NaN  NaN  NaN  NaN   NaN  0.004525   NaN   
3  187104   4.74  0.26  0.4  NaN  NaN  NaN  NaN  NaN   NaN  0.004252   NaN   
4  187105   4.86  0.26  0.4  NaN  NaN  NaN  NaN  NaN   NaN  0.004643   NaN   

   ltr  corpr  svar  csp  CRSP_SPvw  CRSP_SPvwx  
0  NaN    NaN   NaN  NaN        NaN         NaN  
1  NaN    NaN   NaN  NaN        NaN         NaN  
2  NaN    NaN   NaN  NaN        NaN         NaN  
3  NaN    NaN   NaN  NaN        NaN         NaN  
4  NaN    NaN   NaN  NaN        NaN         NaN  


In [34]:
# ====================================================
# Process macro predictors
# ====================================================

if macro_raw is not None:
    # The dataset typically has 'yyyymm' as the date column
    macro = macro_raw.copy()
    
    # Identify date column (could be 'yyyymm', 'Date', etc.)
    date_col = [c for c in macro.columns if 'date' in c.lower() or 'yyyymm' in c.lower()]
    if not date_col:
        date_col = [macro.columns[0]]  # Usually the first column
    date_col = date_col[0]
    
    macro[date_col] = macro[date_col].astype(str)
    macro = macro[macro[date_col].str.len() >= 6].copy()
    macro['date'] = pd.to_datetime(macro[date_col].str[:6], format='%Y%m').dt.to_period('M')
    
    # Construct the 8 predictors used in GKX (2020)
    # Note: exact column names depend on the Goyal dataset version
    
    macro_cols_available = macro.columns.tolist()
    print('Available columns in macro data:')
    print(macro_cols_available)
    
    # Construct predictors (adjust column names as needed)
    macro_predictors = pd.DataFrame()
    macro_predictors['date'] = macro['date']
    
    try:
        # dp: log dividend-price ratio
        macro_predictors['dp'] = np.log(macro['D12'].astype(float)) - np.log(macro['Index'].astype(float))
        # ep: log earnings-price ratio 
        macro_predictors['ep'] = np.log(macro['E12'].astype(float)) - np.log(macro['Index'].astype(float))
        # bm: book-to-market
        macro_predictors['bm'] = macro['b/m'].astype(float)
        # ntis: net equity expansion
        macro_predictors['ntis'] = macro['ntis'].astype(float)
        # tbl: Treasury bill rate
        macro_predictors['tbl'] = macro['tbl'].astype(float)
        # tms: term spread (long-term yield - T-bill)
        macro_predictors['tms'] = macro['lty'].astype(float) - macro['tbl'].astype(float)
        # dfy: default spread (BAA - AAA)
        macro_predictors['dfy'] = macro['BAA'].astype(float) - macro['AAA'].astype(float)
        # svar: stock variance
        macro_predictors['svar'] = macro['svar'].astype(float)
        
        print(f'\nMacro predictors shape: {macro_predictors.shape}')
        print(macro_predictors.describe())
    except KeyError as e:
        print(f'Column not found: {e}')
        print('Please check the column names above and adjust the code accordingly.')
else:
    print('Macro data not loaded. Please download manually (see instructions above).')

macro_predictors.to_csv("data/Macro_Predictors.csv", index=False)



Available columns in macro data:
['yyyymm', 'Index', 'D12', 'E12', 'b/m', 'tbl', 'AAA', 'BAA', 'lty', 'ntis', 'Rfree', 'infl', 'ltr', 'corpr', 'svar', 'csp', 'CRSP_SPvw', 'CRSP_SPvwx', 'date']

Macro predictors shape: (1848, 9)
                dp           ep           bm         ntis          tbl  \
count  1848.000000  1848.000000  1246.000000  1177.000000  1260.000000   
mean     -3.259051    -2.697558     0.543384     0.015331     0.033608   
std       0.466256     0.385916     0.263315     0.025841     0.029543   
min      -4.523640    -4.836478     0.120510    -0.055954     0.000100   
25%      -3.501137    -2.918823     0.323685     0.002195     0.007475   
50%      -3.167254    -2.708179     0.523083     0.015588     0.030400   
75%      -2.930407    -2.447177     0.724794     0.026702     0.050825   
max      -1.873246    -1.670063     2.028478     0.177041     0.163000   

               tms          dfy         svar  
count  1260.000000  1272.000000  1679.000000  
mean      0

---
## 4. Merge Everything & Construct Final Dataset

In [6]:
# ====================================================
#  Set up and & Import data from csv
# ====================================================
import pandas as pd
import numpy as np
import os
from pathlib import Path

# ====================================================
# Paths
# ====================================================
DATA_DIR = Path("data")

# Ajusta estos nombres si tus archivos se llaman distinto
CHARS_FILE = DATA_DIR / "datashare.csv"                # Firm characteristics data 
CRSP_FILE = DATA_DIR / "Monthly_returns_crps.csv"      # CRSP monthly returns from WRDS
MACRO_FILE = DATA_DIR / "Macro_Predictors.csv"         # Macro predictors from Welch & Goyal (2008)

# ====================================================
# Load saved datasets
# ====================================================

# load firm characteristics data
chars = pd.read_csv(CHARS_FILE)
# Basic formatting
chars.rename(columns={'DATE': 'date', 'permno': 'permno'}, inplace=True)
chars['date'] = pd.to_datetime(chars['date'], format='%Y%m%d')
chars['date'] = chars['date'].dt.to_period('M')  # Convert to monthly period

# Load CRSP data of monthly returns
crsp = pd.read_csv(CRSP_FILE)
# Basic formatting
crsp['date'] = pd.PeriodIndex(crsp['date'], freq='M')
crsp = crsp.dropna(subset=['ret_excess'])

# load Macro predictors 
macro_predictors = pd.read_csv(MACRO_FILE)
# Basic formatting
macro_predictors['date'] = pd.PeriodIndex(macro_predictors['date'], freq='M')

print("chars shape:", chars.shape)
print("crsp shape:", crsp.shape)
print("macro_predictors shape:", macro_predictors.shape)


# ====================================================
# Quick checks
# ====================================================
print("\nColumns in chars:")
print(chars.columns.tolist()[:15], "...")

print("\nColumns in crsp:")
print(crsp.columns.tolist())

print("\nColumns in macro_predictors:")
print(macro_predictors.columns.tolist())


chars shape: (4117300, 97)
crsp shape: (3352317, 16)
macro_predictors shape: (1848, 9)

Columns in chars:
['permno', 'date', 'mvel1', 'beta', 'betasq', 'chmom', 'dolvol', 'idiovol', 'indmom', 'mom1m', 'mom6m', 'mom12m', 'mom36m', 'pricedelay', 'turn'] ...

Columns in crsp:
['permno', 'date', 'ret', 'retx', 'prc', 'shrout', 'vol', 'exchcd', 'siccd', 'shrcd', 'dlret', 'dlstcd', 'ret_adj', 'me', 'rf', 'ret_excess']

Columns in macro_predictors:
['date', 'dp', 'ep', 'bm', 'ntis', 'tbl', 'tms', 'dfy', 'svar']


In [7]:
# ====================================================
# Merge characteristics with returns
# ====================================================

# The characteristics are already lagged — the date in datashare.csv
# corresponds to the month whose return is the response variable.
# So we merge on permno and date directly.

Final_data = chars.merge(crsp, on=['permno', 'date'], how='inner')
print(f'After merging chars + returns: {Final_data.shape}')
print(f'Date range: {Final_data["date"].min()} to {Final_data["date"].max()}')
print(f'Unique stocks: {Final_data["permno"].nunique()}')

After merging chars + returns: (3305648, 111)
Date range: 1957-03 to 2021-12
Unique stocks: 25770


In [8]:
# ====================================================
# Merge macro predictors (lagged by 1 month)
# ====================================================

# Lag macro predictors by 1 month (we predict month t returns using month t-1 macro)
macro_predictors_lagged = macro_predictors.copy()
macro_predictors_lagged['date'] = macro_predictors_lagged['date'] + 1  # Shift forward by 1 period

Final_data = Final_data.merge(macro_predictors_lagged, on='date', how='left')
print(f'After merging macro predictors: {Final_data.shape}')

After merging macro predictors: (3305648, 119)


In [9]:
# ====================================================
# Create SIC-2 industry dummies (74 dummies)
# ====================================================

Final_data['sic2'] = (Final_data['siccd'] // 100).astype(int)
industry_dummies = pd.get_dummies(Final_data['sic2'], prefix='ind', dtype=int)

print(f'Number of SIC-2 industry groups: {industry_dummies.shape[1]}')
print(f'Industry dummy columns: {industry_dummies.columns.tolist()[:10]}... (showing first 10)')

Number of SIC-2 industry groups: 90
Industry dummy columns: ['ind_0', 'ind_1', 'ind_2', 'ind_4', 'ind_5', 'ind_7', 'ind_8', 'ind_9', 'ind_10', 'ind_11']... (showing first 10)


In [ ]:
# ====================================================
# Create interaction terms: characteristics × macro variables
# ====================================================

# Identify the 94 characteristic columns
id_cols = ['permno', 'date', 'ret_excess', 'me', 'siccd', 'exchcd', 'sic2']
macro_vars = ['dp', 'ep_y', 'bm_y', 'ntis', 'tbl', 'tms', 'dfy', 'svar']
char_cols = [c for c in Final_data.columns if c not in id_cols + macro_vars]

print(f'Number of firm characteristics: {len(char_cols)}')
print(f'Number of macro variables: {len(macro_vars)}')

# Create interactions
interaction_dfs = []
for macro_var in macro_vars:
    for char_col in char_cols:
        col_name = f'{char_col}_x_{macro_var}'
        interaction_dfs.append(
            pd.Series(Final_data[char_col] * Final_data[macro_var], name=col_name)
        )

interactions = pd.concat(interaction_dfs, axis=1)
print(f'Number of interaction terms: {interactions.shape[1]}')

# Total covariates: 94 chars + (94 × 8 interactions) + 74 industry dummies = 920
total_features = len(char_cols) + interactions.shape[1] + industry_dummies.shape[1]
print(f'\nTotal feature count: {len(char_cols)} + {interactions.shape[1]} + {industry_dummies.shape[1]} = {total_features}')

# Save all to parquet
industry_dummies.to_parquet("data/industry_dummies.parquet", index=False)
Final_data.to_parquet("data/final_data_base.parquet", index=False)
interactions.to_parquet("data/interactions.parquet", index=False)



Number of firm characteristics: 104
Number of macro variables: 8
Number of interaction terms: 832

Total feature count: 104 + 832 + 90 = 1026


---

In [1]:
print("Hello World")

Hello World


In [ ]:
import pandas as pd 
import numpy as np  
import os
from pathlib import Path    

Final_data = pd.read_parquet("data/final_data_base.parquet")
interactions = pd.read_parquet("data/interactions.parquet")
industry_dummies = pd.read_parquet("data/industry_dummies.parquet")
print(Final_data.shape)
print(interactions.shape)
print(industry_dummies.shape) 

(3305648, 119)
(3305648, 832)
(3305648, 90)


In [ ]:
# ====================================================
# Assemble final dataset
# ====================================================

# Identify the 94 characteristic columns
id_cols = ['permno', 'date', 'ret_excess', 'me', 'siccd', 'exchcd', 'sic2']
macro_vars = ['dp', 'ep_y', 'bm_y', 'ntis', 'tbl', 'tms', 'dfy', 'svar']
base_cols = ['permno', 'date', 'ret_excess', 'me', 'exchcd']
char_cols = [c for c in Final_data.columns if c not in id_cols + macro_vars]

# Reduce memory pressure before building the wide final frame.
interactions = interactions.astype('float32', copy=False)
industry_dummies = industry_dummies.astype('int8', copy=False)

Final_Data = pd.concat(
    [
        Final_data[base_cols],
        Final_data[char_cols],
        Final_data[macro_vars],
        interactions,
        industry_dummies,
    ],
    axis=1,
    copy=False,
)

print(f'Final dataset shape: {Final_Data.shape}')
print(f'Date range: {Final_Data["date"].min()} to {Final_Data["date"].max()}')
print(f'Unique stocks: {Final_Data["permno"].nunique()}')
print('Missing values (% by column):')
print((Final_Data.isnull().sum() / len(Final_Data) * 100).describe())
print(f'Memory usage (GB): {Final_Data.memory_usage(deep=True).sum() / 1024**3:.2f}')


---
## 5. Pre-processing (following GKX 2020)

From the paper:
1. **Winsorize** characteristics at 1% and 99% each month
2. **Rank-transform** into [-1, 1] interval cross-sectionally each month
3. **Fill missing** with cross-sectional median

In [ ]:
# ====================================================
# Cross-sectional rank transformation
# ====================================================

feature_cols = char_cols  # Apply to the 94 raw characteristics

def rank_transform(group, cols):
    """Rank-transform characteristics cross-sectionally into [-1, 1]."""
    for col in cols:
        ranks = group[col].rank(method='average', na_option='keep')
        n_valid = group[col].notna().sum()
        if n_valid > 1:
            group[col] = 2 * (ranks - 1) / (n_valid - 1) - 1
        else:
            group[col] = 0
    return group

print('Applying cross-sectional rank transformation...')
print('(This may take several minutes for large datasets)')

# Apply per month
final_data_ranked = final_data.groupby('date', group_keys=False).apply(
    lambda g: rank_transform(g, feature_cols)
)

# Fill remaining NaNs in characteristics with 0 (cross-sectional median after ranking)
final_data_ranked[feature_cols] = final_data_ranked[feature_cols].fillna(0)

print('Rank transformation complete.')
print(f'Shape: {final_data_ranked.shape}')

---
## 6. Train/Validation/Test Split

Following GKX (2020):
- **Training:** 1957–1974 (18 years)
- **Validation:** 1975–1986 (12 years)
- **Test (OOS):** 1987–2016 (30 years)

The training window expands over time (expanding window scheme).

In [ ]:
# ====================================================
# Define sample splits
# ====================================================

train_end   = pd.Period('1974-12', freq='M')
valid_end   = pd.Period('1986-12', freq='M')
test_end    = pd.Period('2021-12', freq='M')  # Or 2021-12 if using updated data

mask_train = final_data_ranked['date'] <= train_end
mask_valid = (final_data_ranked['date'] > train_end) & (final_data_ranked['date'] <= valid_end)
mask_test  = (final_data_ranked['date'] > valid_end) & (final_data_ranked['date'] <= test_end)

print(f'Training set:   {mask_train.sum():>10,} obs  (... to {train_end})')
print(f'Validation set: {mask_valid.sum():>10,} obs  ({train_end+1} to {valid_end})')
print(f'Test set:       {mask_test.sum():>10,} obs   ({valid_end+1} to {test_end})')

---
## 7. Save Processed Data

In [ ]:
# ====================================================
# Save to parquet (efficient storage)
# ====================================================

# Convert Period to timestamp for parquet compatibility
save_data = final_data_ranked.copy()
save_data['date'] = save_data['date'].dt.to_timestamp()

output_path = os.path.join(DATA_DIR, 'gkx2020_panel.parquet')
save_data.to_parquet(output_path, index=False)
print(f'Saved processed panel to: {output_path}')
print(f'File size: {os.path.getsize(output_path) / 1e9:.2f} GB')

In [ ]:
# Also save the raw components separately for flexibility
crsp_clean_save = crsp_clean.copy()
crsp_clean_save['date'] = crsp_clean_save['date'].dt.to_timestamp()
crsp_clean_save.to_parquet(os.path.join(DATA_DIR, 'crsp_monthly_returns.parquet'), index=False)

macro_predictors_save = macro_predictors.copy()
macro_predictors_save['date'] = macro_predictors_save['date'].dt.to_timestamp()
macro_predictors_save.to_parquet(os.path.join(DATA_DIR, 'macro_predictors.parquet'), index=False)

print('All data saved successfully!')
print(f'\nFiles in {DATA_DIR}:')
for f in os.listdir(DATA_DIR):
    size = os.path.getsize(os.path.join(DATA_DIR, f)) / 1e6
    print(f'  {f}: {size:.1f} MB')

In [ ]:
# Close WRDS connection
wrds_db.close()
print('WRDS connection closed.')

---
## Summary

| Component | Source | Observations |
|---|---|---|
| 94 firm characteristics | Dacheng Xiu's website (datashare.zip) | Pre-lagged |
| Monthly excess returns | CRSP via WRDS | Adjusted for delistings |
| 8 macro predictors | Welch & Goyal (2008) | Lagged 1 month |
| 74 industry dummies | SIC-2 codes from CRSP | Cross-sectional |
| Interactions | 94 × 8 = 752 | Char × Macro |
| **Total features** | **920** | As in GKX (2020) |

### Next steps for the replication:
1. Train ML models: OLS, Ridge, Lasso, Elastic Net, PCR, PLS, Random Forest, Gradient Boosting, Neural Networks (NN1-NN5)
2. Evaluate using out-of-sample $R^2_{OOS}$
3. Portfolio analysis: decile sorts based on predicted returns
4. Variable importance analysis